# Bài 0005 — Triton trên GPU thật (Colab T4)

Mục tiêu: chạy đúng pipeline serving → benchmark → metrics đã xây trên CPU, lần này trên **GPU T4** của Colab, thu DCGM metrics, và so sánh **CPU vs GPU theo cost**.

> ⚠️ Path Colab **chưa được verify** — chính là việc của bạn ở bài này. Notebook thiết kế theo kiểu probe: mỗi bước kiểm tra điều kiện rồi mới đi tiếp, và in rõ hướng dẫn rẽ nhánh.


## Bước 0 — Probe: Colab hôm nay có gì?

Chạy cell dưới. Ba kết quả có thể:
- **T4/L4 với driver ≥ r580** → zip release CUDA 13 (path chính của notebook này) khả thi.
- **Driver cũ hơn (< r580)** → zip build cu132 sẽ không init được CUDA; xem md cell sau bước probe để chọn path thay thế (runtime khác, hoặc CPU-only + ghi chú trong báo cáo).
- **Không có GPU** (Runtime → Change runtime type → GPU chưa bật) → bật GPU runtime rồi chạy lại.


In [ ]:
import subprocess, re
!nvidia-smi

driver = 'unknown'
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True)
    driver = out.strip().splitlines()[0]
    major = int(driver.split('.')[0])
    print(f"\nDriver: {driver} (r{major})")
    if major >= 580:
        print('OK — driver hỗ trợ CUDA 13 → zip release khả thi.')
    else:
        print(f'CẢNH BÁO — driver r{major} < r580: build cu132 trong zip sẽ KHÔNG dùng được CUDA.')
        print('Path thay thế: đổi runtime (thường có driver mới hơn), hoặc chạy lại buổi học trên máy CPU + ghi chú vào báo cáo.')
except FileNotFoundError:
    print('KHÔNG có nvidia-smi — runtime này không có GPU. Runtime → Change runtime type → GPU.')


## Bước 1 — Cài Triton (cùng công thức máy CPU, không Docker)

Tái dùng đúng recipe đã verify ở repo: zip release + extract openssl/dcgm (không apt install — `dpkg -x` vào thư mục riêng, trỏ `LD_LIBRARY_PATH`, an toàn với hệ thống Colab) + 2 wheel NVIDIA cho `libcudart/libcupti`.

⚠️ *Trên Colab (Ubuntu 22.04) chưa verify — nếu lỗi link glibc/ssl, chụp log lại (bài học thật) và thử driver deb từ ubuntu2404 như cell dưới đã làm.*


In [ ]:
import os
BASE = '/content/.triton-gpu'
os.makedirs(BASE, exist_ok=True)

# [1/3] Server zip (~1.1GB, build cu132 — GPU sẵn trong build, có driver thì dùng GPU, không thì CPU)
if not os.path.isdir(f'{BASE}/triton_dist/tritonserver'):
    !curl -L --fail -o {BASE}/server.zip 'https://github.com/triton-inference-server/server/releases/download/v2.73.0/tritonserver-2.73.0%2Bnv26.09-cu132-cp312-manylinux_2_34-x86_64.zip'
    !unzip -q {BASE}/server.zip -d {BASE}/triton_dist && rm {BASE}/server.zip
else:
    print('server: đã có')

# [2/3] OpenSSL 3.4 + DCGM (extract, không cài đè hệ thống)
if not os.path.isdir(f'{BASE}/ssl_extract'):
    !curl -sSf -O http://archive.ubuntu.com/ubuntu/pool/main/o/openssl/libssl3t64_3.4.1-1ubuntu4_amd64.deb
    !dpkg-deb -x libssl3t64_3.4.1-1ubuntu4_amd64.deb {BASE}/ssl_extract && rm libssl3t64_3.4.1-1ubuntu4_amd64.deb
    !curl -sSf -O https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/datacenter-gpu-manager-4-core_4.6.1-1_amd64.deb
    !dpkg-deb -x datacenter-gpu-manager-4-core_4.6.1-1_amd64.deb {BASE}/dcgm_extract && rm datacenter-gpu-manager-4-core_4.6.1-1_amd64.deb
else:
    print('ssl/dcgm: đã có')

# [3/3] CUDA runtime + CUPTI wheels
if not os.path.isfile(f'{BASE}/cuda_libs/nvidia/cu13/lib/libcudart.so.13'):
    !pip install -q --target={BASE}/cuda_libs --index-url https://pypi.nvidia.com nvidia-cuda-runtime nvidia-cuda-cupti
else:
    print('cuda libs: đã có')
print('Setup xong.')


## Bước 2 — Lấy lab repo (model repository + scripts đã xây 4 bài trước)


In [ ]:
import os
if not os.path.isdir('/content/triton-serving-lab'):
    !git clone https://github.com/dauvannam1804/triton-serving-lab /content/triton-serving-lab
else:
    !git -C /content/triton-serving-lab pull
LAB = '/content/triton-serving-lab'
!python3 {LAB}/scripts/make_onnx_model.py


## Bước 3 — Model thứ hai: CỠ LỚN (512×512)

`onnx_linear` quá nhỏ — GPU chẳng có gì để làm (kernel launch tốn hơn cả phép tính). Để GPU lộ giá trị, cần model **nặng**: matmul 512×512. Sinh ngay tại chỗ bằng onnx (không tải model nào).


In [ ]:
import numpy as np, onnx, os
from onnx import TensorProto, helper, numpy_helper

DIM = 512
outdir = f'{LAB}/model_repository/onnx_bigmatmul/1'
os.makedirs(outdir, exist_ok=True)

x = helper.make_tensor_value_info('INPUT0', TensorProto.FLOAT, ['batch', DIM])
y = helper.make_tensor_value_info('OUTPUT0', TensorProto.FLOAT, ['batch', DIM])
rng = np.random.default_rng(42)
w = rng.normal(0, 0.02, (DIM, DIM)).astype(np.float32)  # trọng số 'thật' để ORT không triệt tiêu phép tính
b = np.zeros(DIM, dtype=np.float32)
nodes = [helper.make_node('MatMul', ['INPUT0', 'W'], ['mm']), helper.make_node('Add', ['mm', 'B'], ['OUTPUT0'])]
graph = helper.make_graph(nodes, 'onnx_bigmatmul', [x], [y], initializer=[
    numpy_helper.from_array(w, 'W'), numpy_helper.from_array(b, 'B')])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)], producer_name='lab-0005')
model.ir_version = 10
onnx.checker.check_model(model)
onnx.save(model, f'{outdir}/model.onnx')
print('Đã sinh onnx_bigmatmul:', f'{outdir}/model.onnx')


## Bước 4 — Config GPU: chỉ đổi `kind` thành `KIND_GPU`

Model repo giữ nguyên — Triton đọc config lúc load. Cell dưới ghi `config.pbtxt` cho `onnx_bigmatmul` với instance `KIND_GPU`. Lưu ý số instance trên GPU: thường **1** là đủ (nhiều instance cùng GPU chỉ tăng tranh chấp); khác hẳn tư duy CPU.


In [ ]:
cfg = '''name: "onnx_bigmatmul"
platform: "onnxruntime_onnx"
max_batch_size: 32
input [ { name: "INPUT0" data_type: TYPE_FP32 dims: [ 512 ] } ]
output [ { name: "OUTPUT0" data_type: TYPE_FP32 dims: [ 512 ] } ]
instance_group [ { count: 1 kind: KIND_GPU } ]
dynamic_batching {
  preferred_batch_size: [ 8, 16, 32 ]
  max_queue_delay_microseconds: 100
}
'''
open(f'{LAB}/model_repository/onnx_bigmatmul/config.pbtxt', 'w').write(cfg)
print(cfg)


## Bước 5 — Serve (background) + health check

Chạy `tritonserver` như một process nền, log ra file — không có 'terminal 1' trên Colab nên background + log là cách làm việc chuẩn ở đây.


In [ ]:
import os, subprocess, time, urllib.request
BASE = '/content/.triton-gpu'
env = dict(os.environ, LD_LIBRARY_PATH=(
    f'{BASE}/ssl_extract/usr/lib/x86_64-linux-gnu:'
    f'{BASE}/triton_dist/tritonserver/lib64:'
    f'{BASE}/cuda_libs/nvidia/cu13/lib:'
    f'{BASE}/dcgm_extract/usr/lib/x86_64-linux-gnu'))
log = open('/content/tritonserver.log', 'w')
proc = subprocess.Popen([
    f'{BASE}/triton_dist/tritonserver/bin/tritonserver',
    f'--model-repository={LAB}/model_repository',
    '--backend-directory=' + f'{BASE}/triton_dist/tritonserver/backends',
    '--http-port=8000', '--grpc-port=8001', '--metrics-port=8002',
], env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    time.sleep(1)
    try:
        if urllib.request.urlopen('http://localhost:8000/v2/health/ready', timeout=2).status == 200:
            print('READY — server sống (lần probe thứ', _, ')'); break
    except Exception:
        pass
else:
    print('CHƯA READY sau 60s — xem log:  !tail -50 /content/tritonserver.log')
!grep -B1 -A8 'Model.*Version.*Status' /content/tritonserver.log | head -12
!grep -E 'CUDA|GPU' /content/tritonserver.log | head -5


## Bước 6 — Inference kiểm chứng + perf_analyzer

Inference smoke test giống hệt bài 0001 (payload không đổi — client không cần biết server CPU hay GPU, đó là điểm mạnh của kiến trúc này). Sau đó benchmark 2 model: linear (nhỏ) và bigmatmul (nặng), mỗi cái c=1 và c=8.


In [ ]:
!curl -s -X POST localhost:8000/v2/models/onnx_linear/infer -H 'Content-Type: application/json' \
  -d '{"inputs":[{"name":"INPUT0","shape":[1,10],"datatype":"FP32","data":[1,2,3,4,5,6,7,8,9,10]}],"outputs":[{"name":"OUTPUT0"}]}'
!curl -s -X POST localhost:8000/v2/models/onnx_bigmatmul/infer -H 'Content-Type: application/json' \
  -d '{"inputs":[{"name":"INPUT0","shape":[1,512],"datatype":"FP32","data":[' + ','.join(['1.0']*512) + ']}],"outputs":[{"name":"OUTPUT0"}]}' | head -c 120; echo ' …'


In [ ]:
PA = '/content/.triton-gpu/sdk/tritonserver_sdk/install/bin/perf_analyzer'
# SDK zip (~458MB) chứa perf_analyzer
if not os.path.isfile(PA):
    !curl -L --fail -o /content/sdk.zip 'https://github.com/triton-inference-server/server/releases/download/v2.73.0/tritonserver_sdk-2.73.0%2Bnv26.09-cu132-cp312-manylinux_2_34-x86_64.zip'
    !unzip -q /content/sdk.zip 'tritonserver_sdk/install/*' -d /content/.triton-gpu/sdk && rm /content/sdk.zip

for model in ['onnx_linear', 'onnx_bigmatmul']:
    for conc in ['1', '8']:
        print(f'===== {model} c={conc} =====')
        !{PA} -m {model} --concurrency-range {conc} --measurement-interval 3000 --max-trials 15 | grep -E 'Throughput|p50|p99|Inference count|Execution count|Avg request latency'


## Bước 7 — DCGM metrics: GPU đang làm gì dưới tải?

Cùng endpoint `:8002/metrics` giờ có thêm họ `DCGM_*` (chỉ xuất hiện khi có GPU). Chạy collector của lab trong lúc perf_analyzer đạp tải, rồi soi GPU util.


In [ ]:
!curl -s localhost:8002/metrics | grep -E '^DCGM_FI_DEV_(GPU_UTIL|MEM_COPY_UTIL|FB_USED)' | head -5


In [ ]:
# Terminal 'thứ hai' kiểu Colab: collector chạy nền trong khi cell dưới đạp tải
!nohup python3 {LAB}/scripts/collect_metrics.py --duration 20 --model onnx_bigmatmul --out /content/metrics_gpu.csv > /content/collect.log 2>&1 & sleep 1
!{PA} -m onnx_bigmatmul --concurrency-range 8 --measurement-interval 8000 --max-trials 15 | tail -3
!sleep 12 && tail -14 /content/collect.log


## Bước 8 — Chốt: bảng CPU vs GPU theo cost

Điền số bạn đo được vào bảng sau (CPU baseline lấy từ `reports/0001-cpu-baseline-onnx-linear.md` của lab):

| Model | Nền | Throughput (infer/s) | p50 | p99 | Chi phí | infer/s trên mỗi $/h |
|---|---|---|---|---|---|---|
| onnx_bigmatmul | CPU (máy Nam, 8 core) | *điền* | | | 'miễn phí' (điện) | — |
| onnx_bigmatmul | Colab T4 | *điền* | | | ~$0 (free tier, có quota) / ~$0.53 nếu Colab Pro/GCP | *điền* |

**Công thức** (ngôn ngữ chung của dự án Cost-Aware):

```
cost_per_1M_inference = ($/h × 1000000) / (infer/s × 3600)
```

Câu trả lời 'GPU hay CPU' **không có sẵn** — nó = (độ nặng model) × (tải thực) × (giá mỗi nền). Việc của bạn là đo đủ 3 đại lượng này và để bảng quyết định thay cho tranh luận.


## Dọn dẹp


In [ ]:
# Dừng server khi làm xong
!pkill -f tritonserver
print('Server đã dừng.')
